# 01 CDRMIP Pipeline

In [1]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from functools import partial

from cdrmip_extremes import (
    configs, load_pipeline, io_disk, gwl, utils,
    extremes_pipeline, extremes
)


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/42357/status,
Dashboard: /proxy/42357/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37145,Workers: 0
Dashboard: /proxy/42357/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:39099,Total threads: 1
Dashboard: /proxy/38675/status,Memory: 0 B
Nanny: tcp://127.0.0.1:40787,


In [3]:
config = load_pipeline.LoadConfig(
    experiments=[
        load_pipeline.ExperimentConfig(
            experiment_id='1pctCO2-cdr', table_id='day', variable_id='pr',
            anomaly_reference='piControl',grid_label=None,
            overshoot=True, crossing_gwls = [1.5], gwl_match_period = (320,340)
        ),
    ],
    realign_years_overrides={'1pctCO2': (0, 140)}
)

In [4]:
pipe = load_pipeline.LoadingPipeline(config)

In [5]:
pipe.load_raw()

=== Loading 1pctCO2-cdr ===
Searching for files matching: {'experiment_id': '1pctCO2-cdr', 'table_id': 'day', 'variable_id': 'pr', 'grid_label': None, 'member_id': None}
Loading ACCESS-ESM1-5 / r1i1p1f1
  0241-01-01 12:00:00 -> 1000-12-31 12:00:00
Loading CESM2 / r1i1p1f1
  0001-01-01 00:00:00 -> 0200-12-31 00:00:00
Loading CNRM-ESM2-1 / r1i1p1f2
  1990-01-01 12:00:00 -> 2189-12-31 12:00:00
Loading CanESM5 / r1i1p2f1
  1991-01-01 12:00:00 -> 2290-12-31 12:00:00
Loading GFDL-ESM4 / r1i1p1f1
  0141-01-01 12:00:00 -> 0340-12-31 12:00:00
Loading MIROC-ES2L / r1i1p1f2
  1989-01-01 12:00:00 -> 2490-12-31 12:00:00
Loading NorESM2-LM / r1i1p1f1
  0141-01-01 12:00:00 -> 0399-12-31 12:00:00
Loading UKESM1-0-LL / r1i1p1f2
  1990-01-01 12:00:00 -> 2639-12-30 12:00:00


In [6]:
pipe.concatenate_to_parents()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [ACCESS-ESM1-5] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CESM2 1pctCO2-cdr r1i1p1f1 parent
Loading CESM2 1pctCO2 r1i1p1f1
Concatenating 1pctCO2-cdr -> 1pctCO2 [CESM2] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 00:00:00, child relabeled to start 0140-01-01 00:00:00)
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 1pctCO2 r1i1p1f2
Concatenating 1pctCO2-cdr -> 1pctCO2 [CNRM-ESM2-1] via realign_years=(0, 140)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-31 12:00:00, child relabeled to start 0140-01-01 12:00:00)
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 1pctCO2 r1i1p2f1
Concatenating 1pctCO2-c

In [7]:
pipe.load_piControl()

Searching for ACCESS-ESM1-5 1pctCO2-cdr r1i1p1f1 parent
Loading ACCESS-ESM1-5 piControl r1i1p1f1
Searching for CESM2 1pctCO2-cdr r1i1p1f1 parent
Loading CESM2 piControl r1i1p1f1
Searching for CNRM-ESM2-1 1pctCO2-cdr r1i1p1f2 parent
Loading CNRM-ESM2-1 piControl r1i1p1f2
Searching for CanESM5 1pctCO2-cdr r1i1p2f1 parent
Loading CanESM5 piControl r1i1p2f1
Searching for GFDL-ESM4 1pctCO2-cdr r1i1p1f1 parent
Loading GFDL-ESM4 piControl r1i1p1f1
Searching for MIROC-ES2L 1pctCO2-cdr r1i1p1f2 parent
Loading MIROC-ES2L piControl r1i1p1f2
Searching for NorESM2-LM 1pctCO2-cdr r1i1p1f1 parent
Loading NorESM2-LM piControl r1i1p1f1
Searching for UKESM1-0-LL 1pctCO2-cdr r1i1p1f2 parent
Loading UKESM1-0-LL piControl r1i1p1f2


In [8]:
pipe.regrid()

Building regridder for grid (145,) x (192,)
Building regridder for grid (192,) x (288,)
Building regridder for grid (128,) x (256,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (180,) x (288,)
Building regridder for grid (64,) x (128,)
Building regridder for grid (96,) x (144,)
Building regridder for grid (144,) x (192,)


In [9]:
pipe.compute_anomalies()

In [10]:
pipe.load(attribute='anomaly')

## Save Anomalies

In [11]:
pipe.save_tree(
    'anomaly',
    'pr',
    proc_dir = configs.PROC_DIR,
    activity_id = 'CDRMIP',
    category = 'anomaly',
    overwrite=True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_pr_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr/NorESM2-LM/1pctCO2-cdr/NorESM2-LM_1pctCO2-cdr_r1i1p1f1_pr_anomaly.nc
Sa

## Rx1day

In [9]:
regridded_concat = utils.map_nested(utils.fix_units, pipe.regridded_concat)
rx1day = utils.map_nested(extremes.calc_rxnday, regridded_concat)             # n=1 default

In [10]:
rx5day = utils.map_nested(partial(extremes.calc_rxnday, n=5), regridded_concat)
pi_ref = utils.map_nested(
    lambda ds: extremes.calc_rxnday(utils.fix_units(ds)).mean("year"), pipe.regridded_piControl
)
rx1day_anom = extremes.picontrol_anomaly(rx1day, pi_ref)

In [11]:
rx1day_anom

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 117MB
   Dimensions:  (lat: 90, lon: 180, year: 900)
   Coordinates:
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
     * year     (year) int64 7kB 0 1 2 3 4 5 6 7 ... 893 894 895 896 897 898 899
   Data variables:
       rx1day   (year, lat, lon) float64 117MB dask.array<chunksize=(900, 90, 180), meta=np.ndarray>}},
 'CESM2': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 44MB
   Dimensions:  (lat: 90, lon: 180, year: 340)
   Coordinates:
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
     * year     (year) int64 3kB 0 1 2 3 4 5 6 7 ... 333 334 335 336 337 338 339
   Data variables:
       rx1day   (year, lat, lon) float64 44MB dask.array<chunksize=(340, 90, 180), meta=np.ndarra

In [17]:
io_disk.save_tree(
    rx1day_anom,
    configs.PROC_DIR,
    'rx1day',
    activity_id='CDRMIP',
    category='anomaly'
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_rx1day_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_rx1day_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_rx1day_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_rx1day_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_rx1day_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_rx1day_anomaly.nc


## GWL Analysis

In [12]:
crossing_years = io_disk.load_tree(configs.PROC_DIR, 'crossing_years', 'CDRMIP', as_dataarray=True)

In [13]:
pr_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    pipe.anomaly, crossing_years,
)

In [14]:
rx1day_diffs = utils.map_nested(
    lambda ds, gy: gwl.gwl_branch_difference(gwl.extract_gwl_period(ds, gy, window=21)),
    rx1day, crossing_years,
)

In [23]:
io_disk.save_tree(
    pr_diffs,
    configs.PROC_DIR,
    'pr_gwl_differences',
    activity_id='CDRMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_pr_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_pr_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_pr_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_pr_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_pr_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/pr_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_pr_gwl_differences.nc
Sa

In [15]:
io_disk.save_tree(
    rx1day_diffs,
    configs.PROC_DIR,
    'rx1day_gwl_differences',
    activity_id='CDRMIP',
    category=None,
    overwrite=True
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_rx1day_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_rx1day_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_rx1day_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_rx1day_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_rx1day_gwl_differences.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/rx1day_gwl_differences/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1

In [16]:
rx1day_diffs

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 132kB
   Dimensions:  (lat: 90, lon: 180, gwl: 1)
   Coordinates:
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
     * gwl      (gwl) float64 8B 1.5
   Data variables:
       rx1day   (gwl, lat, lon) float64 130kB dask.array<chunksize=(1, 90, 180), meta=np.ndarray>}},
 'CESM2': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 132kB
   Dimensions:  (lat: 90, lon: 180, gwl: 1)
   Coordinates:
     * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
     * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
     * gwl      (gwl) float64 8B 1.5
   Data variables:
       rx1day   (gwl, lat, lon) float64 130kB dask.array<chunksize=(1, 90, 180), meta=np.ndarray>}},
 'CNRM-ESM2-1': {'1pctCO2-cdr': {'r1i1p1f2': <xarray.Dataset> Size: 132kB
   Dimensions:  (l